# TuRBO-m: independent GP models and regional competition

This notebook compares **one** versus **three** trust regions using the same Thompson-sampling implementation. It is a controlled regional baseline, **not** a direct benchmark of the existing `TuRBOStrategy` API. Both configurations start with 15 evaluations and receive the same q-batch budget. Scores are negative benchmark values (larger is better).

## 実装の流れ

各Trust Regionは独立した観測データ・GP・中心点・成功失敗カウンタを持ちます。`run_experiment`は各領域の初期データから`SingleTaskGP`を学習し、`propose_batch`で候補点と`region_ids`を生成します。目的関数を評価した後、`update_regions`が領域別に観測を振り分けます。Restartが発生した領域だけデータを再初期化してGPを再学習します。

**比較条件:** 1領域と3領域の初期評価はどちらも15点、追加評価は2点×8反復です。1領域側もTuRBO-m管理クラスを使うため、既存`TuRBOStrategy`との直接比較ではありません。詳しくは`docs/optimization/turbo-m.md`を参照してください。


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from botorch.test_functions.synthetic import Ackley, Levy, Rosenbrock

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples' / 'turbo_multi_benchmark.py').exists()), None)
if root is None:
    raise RuntimeError('Run this notebook from the robotorchan repository.')
sys.path.insert(0, str(root / 'examples'))
from turbo_multi_benchmark import run_experiment

torch.set_default_dtype(torch.double)


In [ ]:
for problem in (Ackley(dim=2), Rosenbrock(dim=2), Levy(dim=2)):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for n_regions in (1, 3):
        history = run_experiment(
            problem, n_regions=n_regions, q=2, n_steps=8,
            initial_per_region=15 // n_regions, seed=0,
        )
        ax.plot(range(len(history)), history, marker='o', label=f'{n_regions} region(s)')
    ax.set(title=type(problem).__name__, xlabel='BO iteration', ylabel='Best negative objective')
    ax.legend()
    plt.show()
